# 🔤 Character-Level Tokenization - Back to Basics

## What is Character Tokenization?

**Character tokenization treats each character as a separate token.**

### The Simplest Approach:

```python
"Hello" → ['H', 'e', 'l', 'l', 'o']
```

That's it! Each character becomes a token.

### Why Character Tokenization? 🤔

✅ **Tiny vocabulary**: Only ~100-300 characters
✅ **No unknown words**: Can represent ANY text
✅ **Simple**: No complex algorithms needed
✅ **Good for**: Spell correction, character-level tasks

❌ **But...**
- Way too many tokens (inefficient)
- Loses word-level meaning
- Longer sequences = harder to learn
- Not used in modern RAG systems

## When to Use Character Tokenization?

1. **Spell checking/correction**: Character patterns matter
2. **DNA/protein sequences**: Character-level is natural
3. **Low-resource languages**: When you have very little data
4. **Character-level predictions**: Next character generation
5. **Understanding fundamentals**: Great for learning!

## The Trade-off:

```
Word-level:     "Hello World" → 2 tokens, 50k+ vocab
Character-level: "Hello World" → 11 tokens, ~100 vocab
Subword (BPE):  "Hello World" → 3 tokens, 50k vocab ✅ Best!
```

## 1. Basic Character Tokenization

In [1]:
# Simplest implementation
text = "Hello, World!"

# Method 1: Direct list conversion
char_tokens = list(text)

print(f"Text: {text}")
print(f"Character tokens: {char_tokens}")
print(f"Number of tokens: {len(char_tokens)}")
print(f"\nVocabulary (unique chars): {sorted(set(char_tokens))}")

Text: Hello, World!
Character tokens: ['H', 'e', 'l', 'l', 'o', ',', ' ', 'W', 'o', 'r', 'l', 'd', '!']
Number of tokens: 13

Vocabulary (unique chars): [' ', '!', ',', 'H', 'W', 'd', 'e', 'l', 'o', 'r']


## 2. Building a Character Vocabulary

In [2]:
# Create vocabulary from corpus
corpus = [
    "The quick brown fox",
    "jumps over the lazy dog",
    "Natural language processing"
]

# Collect all characters
all_chars = set()
for text in corpus:
    all_chars.update(text)

# Create char to ID mapping
char2id = {char: idx for idx, char in enumerate(sorted(all_chars))}
id2char = {idx: char for char, idx in char2id.items()}

print(f"Vocabulary size: {len(char2id)}")
print(f"\nCharacter to ID mapping:")
for char, idx in sorted(char2id.items()):
    print(f"'{char}' → {idx}")

Vocabulary size: 29

Character to ID mapping:
' ' → 0
'N' → 1
'T' → 2
'a' → 3
'b' → 4
'c' → 5
'd' → 6
'e' → 7
'f' → 8
'g' → 9
'h' → 10
'i' → 11
'j' → 12
'k' → 13
'l' → 14
'm' → 15
'n' → 16
'o' → 17
'p' → 18
'q' → 19
'r' → 20
's' → 21
't' → 22
'u' → 23
'v' → 24
'w' → 25
'x' → 26
'y' → 27
'z' → 28


## 3. Encoding and Decoding

In [3]:
class CharacterTokenizer:
    def __init__(self):
        self.char2id = {}
        self.id2char = {}
        
    def build_vocab(self, texts):
        """Build vocabulary from texts"""
        # Collect all unique characters
        chars = set()
        for text in texts:
            chars.update(text)
        
        # Add special tokens
        chars.update(['<PAD>', '<UNK>', '<SOS>', '<EOS>'])
        
        # Create mappings
        self.char2id = {char: idx for idx, char in enumerate(sorted(chars))}
        self.id2char = {idx: char for char, idx in self.char2id.items()}
        
    def encode(self, text):
        """Convert text to IDs"""
        return [self.char2id.get(char, self.char2id['<UNK>']) for char in text]
    
    def decode(self, ids):
        """Convert IDs back to text"""
        return ''.join([self.id2char.get(idx, '<UNK>') for idx in ids])
    
    def vocab_size(self):
        return len(self.char2id)

# Create and train tokenizer
tokenizer = CharacterTokenizer()
tokenizer.build_vocab(corpus)

# Test encoding
test_text = "Hello RAG!"
encoded = tokenizer.encode(test_text)
decoded = tokenizer.decode(encoded)

print(f"Text: {test_text}")
print(f"Encoded: {encoded}")
print(f"Decoded: {decoded}")
print(f"\nVocabulary size: {tokenizer.vocab_size()}")

Text: Hello RAG!
Encoded: [4, 11, 18, 18, 21, 0, 4, 4, 4, 4]
Decoded: <UNK>ello <UNK><UNK><UNK><UNK>

Vocabulary size: 33


## 4. Comparison with Other Methods

In [4]:
# Compare sequence lengths
from transformers import GPT2Tokenizer, BertTokenizer

# Load tokenizers
gpt2_tok = GPT2Tokenizer.from_pretrained('gpt2')
bert_tok = BertTokenizer.from_pretrained('bert-base-uncased')

test_sentences = [
    "Hello",
    "Natural language processing",
    "Retrieval-Augmented Generation systems are powerful"
]

print("Token Count Comparison:\n")
print(f"{'Text':<50} {'Char':<8} {'BPE':<8} {'WordPiece'}")
print("="*80)

for text in test_sentences:
    char_count = len(list(text))
    bpe_count = len(gpt2_tok.tokenize(text))
    wp_count = len(bert_tok.tokenize(text))
    
    print(f"{text:<50} {char_count:<8} {bpe_count:<8} {wp_count}")

print("\n💡 Character tokenization creates WAY more tokens!")

Token Count Comparison:

Text                                               Char     BPE      WordPiece
Hello                                              5        1        1
Natural language processing                        27       3        3
Retrieval-Augmented Generation systems are powerful 51       10       7

💡 Character tokenization creates WAY more tokens!


## 5. Character-Level RNN (Classic Approach)

In [5]:
# Prepare data for character-level model
text = "hello world this is character level tokenization"

# Build vocabulary
chars = sorted(set(text))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

print(f"Text: {text}")
print(f"Unique characters: {len(chars)}")
print(f"Characters: {chars}")

# Create training sequences (predict next character)
seq_length = 10
sequences = []
next_chars = []

for i in range(len(text) - seq_length):
    sequences.append(text[i:i+seq_length])
    next_chars.append(text[i+seq_length])

print(f"\nTraining sequences: {len(sequences)}")
print(f"\nExample:")
for i in range(3):
    print(f"Input: '{sequences[i]}' → Target: '{next_chars[i]}'")

Text: hello world this is character level tokenization
Unique characters: 17
Characters: [' ', 'a', 'c', 'd', 'e', 'h', 'i', 'k', 'l', 'n', 'o', 'r', 's', 't', 'v', 'w', 'z']

Training sequences: 38

Example:
Input: 'hello worl' → Target: 'd'
Input: 'ello world' → Target: ' '
Input: 'llo world ' → Target: 't'


## 6. Vocabulary Size Analysis

In [6]:
import string

# Different language character sets
character_sets = {
    "ASCII lowercase": list(string.ascii_lowercase),
    "ASCII letters": list(string.ascii_letters),
    "Alphanumeric": list(string.ascii_letters + string.digits),
    "Printable ASCII": list(string.printable),
    "Extended (Latin-1)": list(chr(i) for i in range(256)),
}

print("Character Vocabulary Sizes:\n")
print(f"{'Character Set':<25} {'Size':<10} {'Sample'}")
print("="*70)

for name, chars in character_sets.items():
    sample = ''.join(chars[:20]) if len(chars) > 20 else ''.join(chars)
    print(f"{name:<25} {len(chars):<10} {sample[:30]}...")

print("\n💡 Even large character sets are tiny compared to word vocabularies!")

Character Vocabulary Sizes:

Character Set             Size       Sample
ASCII lowercase           26         abcdefghijklmnopqrst...
ASCII letters             52         abcdefghijklmnopqrst...
Alphanumeric              62         abcdefghijklmnopqrst...
Printable ASCII           100        0123456789abcdefghij...
Extended (Latin-1)        256         	
...

💡 Even large character sets are tiny compared to word vocabularies!


## 7. Handling Special Characters and Unicode

In [7]:
# Multilingual character tokenization
multilingual_texts = [
    ("English", "Hello World"),
    ("French", "Bonjour le monde"),
    ("Spanish", "¡Hola Mundo!"),
    ("Japanese", "こんにちは"),
    ("Chinese", "你好世界"),
    ("Emoji", "Hello 👋 World 🌍"),
]

print("Character Tokenization Across Languages:\n")
print(f"{'Language':<12} {'Text':<25} {'Chars':<8} {'Unique'}")
print("="*70)

for lang, text in multilingual_texts:
    chars = list(text)
    unique_chars = set(chars)
    print(f"{lang:<12} {text:<25} {len(chars):<8} {len(unique_chars)}")

print("\n🌍 Character tokenization works for ANY language!")

Character Tokenization Across Languages:

Language     Text                      Chars    Unique
English      Hello World               11       8
French       Bonjour le monde          16       11
Spanish      ¡Hola Mundo!              12       11
Japanese     こんにちは                     5        5
Chinese      你好世界                      4        4
Emoji        Hello 👋 World 🌍           15       10

🌍 Character tokenization works for ANY language!


## 8. Practical Use Case: Spell Correction

In [8]:
# Character-level features for spell checking
def char_ngrams(text, n=3):
    """Extract character n-grams"""
    return [text[i:i+n] for i in range(len(text)-n+1)]

# Example: Detect misspellings using character patterns
correct_word = "algorithm"
misspelled_word = "algoritm"  # missing 'h'

correct_trigrams = char_ngrams(correct_word, 3)
misspelled_trigrams = char_ngrams(misspelled_word, 3)

print(f"Correct word: {correct_word}")
print(f"Character 3-grams: {correct_trigrams}")

print(f"\nMisspelled word: {misspelled_word}")
print(f"Character 3-grams: {misspelled_trigrams}")

# Find missing/different n-grams
missing = set(correct_trigrams) - set(misspelled_trigrams)
extra = set(misspelled_trigrams) - set(correct_trigrams)

print(f"\nMissing n-grams: {missing}")
print(f"Extra n-grams: {extra}")
print("\n💡 Character n-grams help detect spelling errors!")

Correct word: algorithm
Character 3-grams: ['alg', 'lgo', 'gor', 'ori', 'rit', 'ith', 'thm']

Misspelled word: algoritm
Character 3-grams: ['alg', 'lgo', 'gor', 'ori', 'rit', 'itm']

Missing n-grams: {'thm', 'ith'}
Extra n-grams: {'itm'}

💡 Character n-grams help detect spelling errors!


## 9. Memory Efficiency Comparison

In [9]:
# Calculate memory requirements
import sys

# Sample text
text = "Natural language processing with deep learning" * 100

# Character tokenization
char_tokens = list(text)
char_vocab_size = len(set(char_tokens))
char_sequence_length = len(char_tokens)

# Word tokenization
word_tokens = text.split()
word_vocab_size = len(set(word_tokens))
word_sequence_length = len(word_tokens)

print("Memory and Efficiency Comparison:\n")
print(f"Text length: {len(text)} characters\n")

print(f"{'Metric':<30} {'Character':<15} {'Word'}")
print("="*60)
print(f"{'Vocabulary size':<30} {char_vocab_size:<15} {word_vocab_size}")
print(f"{'Sequence length':<30} {char_sequence_length:<15} {word_sequence_length}")
print(f"{'Compression ratio':<30} {1.0:<15.2f} {char_sequence_length/word_sequence_length:.2f}")

print("\n💡 Character tokenization: Small vocab but LONG sequences!")

Memory and Efficiency Comparison:

Text length: 4600 characters

Metric                         Character       Word
Vocabulary size                18              7
Sequence length                4600            501
Compression ratio              1.00            9.18

💡 Character tokenization: Small vocab but LONG sequences!


## 10. Byte-Level Character Tokenization

In [10]:
# Byte-level encoding (always 256 vocab size)
def byte_encode(text):
    """Encode text as bytes"""
    return list(text.encode('utf-8'))

def byte_decode(byte_list):
    """Decode bytes back to text"""
    return bytes(byte_list).decode('utf-8', errors='ignore')

# Test on different texts
test_texts = [
    "Hello",
    "Hello 🌍",
    "你好",
]

print("Byte-Level Tokenization:\n")
print(f"{'Text':<15} {'Bytes':<30} {'Vocab Size'}")
print("="*70)

for text in test_texts:
    bytes_list = byte_encode(text)
    vocab_size = 256  # Always 256 for byte-level
    print(f"{text:<15} {str(bytes_list):<30} {vocab_size}")

print("\n💡 Byte-level always has exactly 256 possible tokens!")

Byte-Level Tokenization:

Text            Bytes                          Vocab Size
Hello           [72, 101, 108, 108, 111]       256
Hello 🌍         [72, 101, 108, 108, 111, 32, 240, 159, 140, 141] 256
你好              [228, 189, 160, 229, 165, 189] 256

💡 Byte-level always has exactly 256 possible tokens!


## 11. Why Modern RAG Doesn't Use Character Tokenization

In [11]:
# Demonstrate the inefficiency
rag_document = """Retrieval-Augmented Generation (RAG) is a technique that combines 
information retrieval with text generation. It helps language models access external 
knowledge and provide more accurate, factual responses."""

# Character tokenization
char_tokens = list(rag_document)

# Subword tokenization (BPE)
bpe_tokens = gpt2_tok.tokenize(rag_document)

print("RAG Document Tokenization:\n")
print(f"Document: {rag_document[:60]}...\n")

print(f"Character tokens: {len(char_tokens)}")
print(f"BPE tokens: {len(bpe_tokens)}")
print(f"\nEfficiency ratio: {len(char_tokens)/len(bpe_tokens):.1f}x more tokens with characters!")

print("\n❌ Problems with character tokenization for RAG:")
print("  1. 5-10x more tokens → slower processing")
print("  2. Longer sequences → more memory")
print("  3. Loses word-level meaning")
print("  4. Harder to learn semantic relationships")
print("\n✅ Solution: Use subword tokenization (BPE/WordPiece/SentencePiece)!")

RAG Document Tokenization:

Document: Retrieval-Augmented Generation (RAG) is a technique that com...

Character tokens: 208
BPE tokens: 40

Efficiency ratio: 5.2x more tokens with characters!

❌ Problems with character tokenization for RAG:
  1. 5-10x more tokens → slower processing
  2. Longer sequences → more memory
  3. Loses word-level meaning
  4. Harder to learn semantic relationships

✅ Solution: Use subword tokenization (BPE/WordPiece/SentencePiece)!


## Key Takeaways 🎯

### ✅ Character Tokenization Pros:

1. **Tiny vocabulary**: 100-300 tokens max
2. **No unknown words**: Can represent anything
3. **Simple**: Easy to implement
4. **Universal**: Works for any language
5. **Good for**: Spell checking, character-level tasks

### ❌ Character Tokenization Cons:

1. **Too many tokens**: 5-10x more than subword
2. **Loses meaning**: No word-level semantics
3. **Inefficient**: More memory, slower processing
4. **Harder to learn**: Longer context needed
5. **Not used in production RAG**: Subword is better

### 📊 Quick Comparison:

```python
Text: "Retrieval-Augmented Generation"

Character:  33 tokens, vocab=20
Word-level:  3 tokens, vocab=3 (but can't handle unknown)
Subword:     5 tokens, vocab=50k ✅ BEST!
```

### 🎯 When to Use Character Tokenization:

**Use it for:**
- Spell checking/correction
- Character-level language modeling
- Very low-resource languages
- DNA/protein sequence analysis
- Learning fundamentals

**Don't use it for:**
- ❌ Modern RAG systems
- ❌ Large-scale NLP
- ❌ Embedding generation
- ❌ Production systems

### 💡 The Evolution:

```
Character → Word → Subword (BPE/WordPiece/SentencePiece)
   ↓          ↓           ↓
Too many   Too big    Just right! ✅
 tokens    vocab
```

### 🚀 For RAG Systems:

**The verdict:**
- Character tokenization is NOT recommended
- Use BPE (GPT), WordPiece (BERT), or SentencePiece (T5)
- Subword methods give the best balance

### 📚 Historical Note:

Character-level models were popular in early deep learning (2015-2017) but have been largely replaced by subword tokenization for most tasks. They remain relevant for specific use cases like spell checking and character-level modeling.

## Next Steps 🔜

Move to `06_Sentence_Tokenization.ipynb` for document chunking!

That's where we learn to split documents for RAG! 📄✂️